# Imports

In [ ]:
# Imports
import numpy as np
from numpy import nan
import pandas as pd
import matplotlib.pyplot as plt
import os
import scipy
import scipy.stats as stats
from scipy.spatial.distance import cdist
import itertools
from tqdm.auto import tqdm
import seaborn as sns
import importlib
from scipy.stats import bernoulli
import warnings
from GP import *
from plotter import *
from scipy.special import softmax
from scipy.spatial.distance import cdist
import gymnasium as gym
from gymnasium.envs.registration import register, registry, make, spec
import pickle
import copy
from itertools import product
import json
from functools import partial
from scipy.optimize import Bounds, minimize, differential_evolution
import multiprocess as mp
from pybads import BADS
import pymer4.models as pm

from utils import make_env, Node, Tree, argm, data_keys, grid_keys, parse_lists, KL_divergence, profile_func, KL_sim, value_iteration, load_data, rotate_grid_world, save_rotated_data, generate_ppt_sequence, loocv_split
from MCTS import MonteCarloTreeSearch, MonteCarloTreeSearch_Free, MonteCarloTreeSearch_AFC

import IPython

from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy import dmatrix

import multiprocess as mp
import pingouin as pg
from scipy.special import expit

from agents import Farmer
from samplers import GridSampler


warnings.filterwarnings('ignore')


%load_ext autoreload
%autoreload 2

### Main trials

In [ ]:
## env inits
N = 9
metric = 'cityblock'
known_costs = False
beta_params = {
    'alpha_row': 0.25,
    'beta_row': 0.25,
    'alpha_col': 0.25,
    'beta_col': 0.25
    # 'alpha_row': 0.5,
    # 'beta_row': 0.5,
    # 'alpha_col': 10,
    # 'beta_col': 0.1
    }

## trial info
n_participants = 20
n_cities = 6
n_days = 5
n_trials = 4
expt = 'AFC'
n_afc = 2
expt_info = {
    'type': expt,
    'n_afc': n_afc,
}
save_path = 'expt/assets/trial_sequences/expt_2'
parallel = True

## serial
if not parallel:
    print('Generating {} experiment sequences serially'.format(n_participants))
    for p in range(1,n_participants+1):
        generate_ppt_sequence(p, n_cities, n_days, n_trials, expt_info.copy(), beta_params, metric, n_afc, N, save_path)

## parallel
elif parallel:
    if __name__ == '__main__':
        print('Generating {} experiment sequences in parallel'.format(n_participants))
        n_cores = min(mp.cpu_count(), n_participants)
        with mp.Pool(n_cores) as pool:
            results = list(tqdm(pool.starmap(
                generate_ppt_sequence,
                [(p, n_cities, n_days, n_trials, expt_info.copy(), beta_params, metric, n_afc, N, save_path)
                for p in range(1, n_participants + 1)]
            ), total=n_participants))


Generating 20 experiment sequences in parallel


  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
### some sanity checks

## load every sequence 
path = '/Users/grahamadmin/Documents/context_exploration/expt/assets/trial_sequences/expt_2/expt_info'
path = '/Users/grahamadmin/Documents/context_exploration/useful_saves/expt_optimisation/simulated_envs/expt_info'
proportions = []
for file in os.listdir(path):
    if file.endswith('.json'):
        with open(os.path.join(path, file), 'r') as f:
            seq = json.load(f)

        # with open(os.path.join(path, file), 'rb') as f:
        #     seq = pickle.load(f)
        
        ## convert trial info to dataframe
        expt_dict = seq['trial_info']
        df_expt = pd.DataFrame(expt_dict)

        ## debugging: see how many days consistently have vertical (or horizontal) as dominant axis for all trials
        n_same = 0
        for c in range(n_cities):
            expt_city = df_expt[df_expt['city']==c+1]
            for d in range(n_days):
                expt_day = expt_city[(expt_city['grid']==d+1) & (expt_city['trial']>1)]
                if expt_day['dominant_axis_A'].nunique() == 1:
                    n_same += 1
        proportion = n_same / (n_cities * n_days)
        proportions.append(proportion)
        print(f'{n_same} days with consistent dominant axis across trials ({proportion*100:.2f}%)')

print()
print(f'Average proportion across participants: {np.mean(proportions)*100:.2f}%')

2 days with consistent dominant axis across trials (20.00%)
2 days with consistent dominant axis across trials (20.00%)
3 days with consistent dominant axis across trials (30.00%)
5 days with consistent dominant axis across trials (50.00%)
1 days with consistent dominant axis across trials (10.00%)
1 days with consistent dominant axis across trials (10.00%)
3 days with consistent dominant axis across trials (30.00%)
1 days with consistent dominant axis across trials (10.00%)
4 days with consistent dominant axis across trials (40.00%)
2 days with consistent dominant axis across trials (20.00%)
2 days with consistent dominant axis across trials (20.00%)
3 days with consistent dominant axis across trials (30.00%)
4 days with consistent dominant axis across trials (40.00%)
4 days with consistent dominant axis across trials (40.00%)
4 days with consistent dominant axis across trials (40.00%)
2 days with consistent dominant axis across trials (20.00%)
3 days with consistent dominant axis acr

#### Rotate funcs


In [ ]:
## test with json
# with open('expt/assets/rotated_trial_sequences/rotate_test.json', 'r') as f:
with open('expt/assets/to_rotate/0i30zpvykjyk5btzylrbcfjk.json', 'r') as f:
    example_data = json.load(f)

## or, test with env object
p_n = 416
with open('expt/assets/trial_sequences/env_objects/env_objects_{}.pkl'.format(p_n), 'rb') as f:
    example_data = pickle.load(f)


print(example_data)

# Rotate clockwise
rotated_clockwise = rotate_grid_world(example_data, "clockwise")

# Rotate counter-clockwise
rotated_counter_clockwise = rotate_grid_world(example_data, "counter_clockwise")

# Pretty print function for cost grids
def print_pretty_grid(grid):
    for row in grid:
        print(" ".join("0" if cell == 0 else "X" for cell in row))

# Display results 
if 'sequence' in example_data:
    print("ORIGINAL PATH A:", example_data['sequence']["trial_info"][2]["path_A"])
    print("CLOCKWISE PATH A:", rotated_clockwise['sequence']["trial_info"][2]["path_A"])
    print("COUNTER-CLOCKWISE PATH A:", rotated_counter_clockwise['sequence']["trial_info"][2]["path_A"])

    print("\nORIGINAL COST GRID:")
    print_pretty_grid(example_data['sequence']["env_costs"]["city_1_grid_1"])

    print("\nCLOCKWISE COST GRID:")
    print_pretty_grid(rotated_clockwise['sequence']["env_costs"]["city_1_grid_1"])

    print("\nCOUNTER-CLOCKWISE COST GRID:")
    print_pretty_grid(rotated_counter_clockwise['sequence']["env_costs"]["city_1_grid_1"])

    display(example_data['sequence']['trial_info'][0])
    display(rotated_clockwise['sequence']['trial_info'][0])

    # Example of saving to file
    # save_rotated_data['sequence'](rotated_clockwise, "rotated_clockwise.json")
    # save_rotated_data['sequence'](rotated_counter_clockwise, "rotated_counter_clockwise.json")



{'city_1_grid_1_env_object': [<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridEnv<grids/GridEnv-v0>>>>>], 'city_1_grid_2_env_object': [<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridEnv<grids/GridEnv-v0>>>>>], 'city_1_grid_3_env_object': [<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridEnv<grids/GridEnv-v0>>>>>], 'city_1_grid_4_env_object': [<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridEnv<grids/GridEnv-v0>>>>>], 'city_1_grid_5_env_object': [<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridEnv<grids/GridEnv-v0>>>>>], 'city_2_grid_1_env_object': [<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridEnv<grids/GridEnv-v0>>>>>], 'city_2_grid_2_env_object': [<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridEnv<grids/GridEnv-v0>>>>>], 'city_2_grid_3_env_object': [<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridEnv<grids/GridEnv-v0>>>>>], 'city_2_grid_4_env_object': [<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridEnv<grids/GridEnv-v0>>>>>], 'city_2_grid_5_env_object': [<TimeLimit<OrderEnforcing

#### Rotate collected Ps

In [ ]:
df

,pid,trial,city,path_chosen,button_pressed,reaction_time_ms,context,grid,path_A_expected_cost,path_B_expected_cost,...,prev_observed_cost,path_quality,prev_path_quality,day_cost,path_len,switched_axis,past_overlaps_diff,observed_costs_diff,observed_no_costs_diff,CE_human_consistent
0,0uedgbc95ul43lldhlhia0sp,1.0,1.0,b,j,NaN,row,NaN,-3.329953,-5.728447,...,NaN,-0.666667,NaN,0.0,6.0,NaN,0.0,0.0,0.0,False
1,0uedgbc95ul43lldhlhia0sp,2.0,1.0,a,j,NaN,row,NaN,-4.034815,-0.328480,...,-4.0,-0.500000,-0.666667,-4.0,6.0,True,1.0,0.0,1.0,True
2,0uedgbc95ul43lldhlhia0sp,3.0,1.0,a,f,NaN,row,NaN,-0.238711,-4.036578,...,-3.0,0.000000,-0.500000,-7.0,7.0,True,0.0,1.0,-1.0,True
3,0uedgbc95ul43lldhlhia0sp,4.0,1.0,b,f,NaN,row,NaN,-4.681725,-7.288686,...,0.0,-0.875000,0.000000,-7.0,8.0,False,1.0,0.0,1.0,False
4,0uedgbc95ul43lldhlhia0sp,1.0,1.0,a,j,NaN,row,NaN,-2.987540,-3.428826,...,NaN,-0.500000,NaN,0.0,8.0,True,0.0,0.0,0.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3195,qo8us8c4p5p65obrsnleudst,4.0,8.0,a,f,NaN,column,NaN,-0.049272,-3.086926,...,-4.0,0.000000,-0.571429,-12.0,6.0,False,1.0,0.0,1.0,False
3196,qo8us8c4p5p65obrsnleudst,1.0,8.0,b,f,NaN,column,NaN,-2.622922,-5.978654,...,NaN,-1.000000,NaN,0.0,7.0,False,0.0,0.0,0.0,False
3197,qo8us8c4p5p65obrsnleudst,2.0,8.0,a,f,NaN,column,NaN,-2.264148,-2.760571,...,-7.0,-0.125000,-1.000000,-7.0,8.0,False,1.0,1.0,0.0,True
3198,qo8us8c4p5p65obrsnleudst,3.0,8.0,a,j,NaN,column,NaN,-0.825894,-1.827877,...,-1.0,-0.166667,-0.125000,-8.0,6.0,False,1.0,1.0,0.0,False


In [ ]:
## get pids to rotate
expt_info_filenames = df['expt_info_filename'].unique()

## gather all the env_object files to the to_rotate folder
for id in expt_info_filenames:

    ## load expt_info file
    with open('expt/assets/trial_sequences/expt_info_{}.json'.format(id), 'r') as f:
        expt_info = json.load(f)

    ## save to expt/assets/to_rotate as is
    with open('expt/assets/to_rotate/expt_info_{}.json'.format(id), 'w') as f:
        json.dump(expt_info, f, indent=4)

        
    
    ## load env_object file
    # with open('expt/assets/trial_sequences/env_objects/env_objects_{}.pkl'.format(id), 'rb') as f:
    #     envs = pickle.load(f)

    ## save to expt/assets/to_rotate


In [ ]:
## get pids to rotate
pids = df['pid'].unique()
# pids = ['e248nl43jdfwg8bisl7sjezc']

## loop through 'expt/assets/to_rotate'. If the filename contains a pid that is in this list, save to a list
files_to_rotate = []
for pid in pids:
    filename = 'expt/assets/to_rotate/{}.json'.format(pid)
    if os.path.exists(filename):
        files_to_rotate.append(filename)
    else:
        print(f"{filename} does not exist")
print('need to rotate ', len(files_to_rotate), 'files')

## rotate each file
for filename in files_to_rotate:
    with open(filename, 'r') as f:
        data = json.load(f)

    ## get the original name of the sequence
    og_filename = data['filename']
    og_id = og_filename[10:-5]

    ## rotate the json
    rotated_data = rotate_grid_world(data, "clockwise")

    ## rotate the original env object
    with open('expt/assets/trial_sequences/env_objects/env_objects_{}.pkl'.format(og_id), 'rb') as f:
        env_object = pickle.load(f)
    rotated_env_object = rotate_grid_world(env_object, "clockwise")

    ## save json
    output_filename = f"expt/assets/rotated_sequences/{og_filename[:-5]}_rotated.json"
    # print(output_filename)
    save_rotated_data(rotated_data['sequence'], output_filename)

    ## save rotated env object
    output_env_filename = f"expt/assets/rotated_sequences/rotated_env_objects/env_objects_{og_id}_rotated.pkl"
    # print(output_env_filename)
    with open(output_env_filename, 'wb') as f:
        pickle.dump(rotated_env_object, f)




need to rotate  1 files
Rotated data saved to expt/assets/rotated_sequences/expt_info_270_rotated.json


#### Rotate all Ps 

In [ ]:
rotated_data['trial_info']

[{'city': 1,
  'context': 'column',
  'grid': 1,
  'trial': 1,
  'start_A': [0, 1],
  'start_B': [1, 7],
  'goal_A': [0, 7],
  'goal_B': [7, 7],
  'path_A': [[0, 1], [0, 2], [0, 3], [0, 4], [0, 5], [0, 6], [0, 7]],
  'path_B': [[1, 7], [2, 7], [3, 7], [4, 7], [5, 7], [6, 7], [7, 7]],
  'path_A_expected_cost': -4.091267804245039,
  'path_B_expected_cost': -6.908312698951509,
  'path_A_actual_cost': -5,
  'path_B_actual_cost': -7,
  'path_A_future_overlap': 0,
  'path_B_future_overlap': 0,
  'abstract_sequence_A': [0, 6],
  'abstract_sequence_B': [6, 0],
  'better_path': 'a',
  'dominant_axis_A': 'horizontal',
  'dominant_axis_B': 'vertical',
  'better_axis': 'horizontal'},
 {'city': 1,
  'context': 'column',
  'grid': 1,
  'trial': 2,
  'start_A': [6, 1],
  'start_B': [2, 5],
  'goal_A': [6, 6],
  'goal_B': [7, 5],
  'path_A': [[6, 1], [6, 2], [6, 3], [6, 4], [6, 5], [6, 6]],
  'path_B': [[2, 5], [3, 5], [4, 5], [5, 5], [6, 5], [7, 5]],
  'path_A_expected_cost': -3.104365990109109,
  'p

In [ ]:
### JUST DONE ENV OBJECTS FOR NOW!

## or, just rotate every env object in expt/assets/trial_sequences/env_objects/...
# env_objects_dir = 'expt/assets/trial_sequences/env_objects/'
# rotated_env_objects_dir = 'expt/assets/trial_sequences/rotated_env_objects/'

# env_objects_files = [f for f in os.listdir(env_objects_dir) if f.endswith('.pkl')]
# for filename in env_objects_files:
#     with open(os.path.join(env_objects_dir, filename), 'rb') as f:
#         env_object = pickle.load(f)

#     ## rotate the env object
#     rotated_env_object = rotate_grid_world(env_object, "clockwise")

#     ## save rotated env object
#     output_env_filename = os.path.join(rotated_env_objects_dir, filename[:-4] + '_rotated.pkl')
#     # print(output_env_filename)
#     with open(output_env_filename, 'wb') as f:
#         pickle.dump(rotated_env_object, f)


## or, just rotate every trial sequence in expt/assets/trial_sequences/
trial_sequences_dir = 'expt/assets/trial_sequences/'
rotated_trial_sequences_dir = 'expt/assets/rotated_trial_sequences/'
trial_sequence_files = [f for f in os.listdir(trial_sequences_dir) if f.endswith('.json') and not f.startswith('practice')]

for filename in trial_sequence_files:
    with open (os.path.join(trial_sequences_dir, filename), 'r', encoding='utf-8') as f:
        # with open(filename, 'r') as f:
        data = json.load(f)

    ## rotate the json
    rotated_data = rotate_grid_world(data, "clockwise")

    # ## save json
    output_filename = f"expt/assets/trial_sequences/rotated_sequences/{filename[:-5]}_rotated.json"
    save_rotated_data(rotated_data, output_filename)

expt/assets/trial_sequences/rotated_sequences/expt_info_444_rotated.json
expt/assets/trial_sequences/rotated_sequences/expt_info_32_rotated.json
expt/assets/trial_sequences/rotated_sequences/expt_info_151_rotated.json
expt/assets/trial_sequences/rotated_sequences/expt_info_278_rotated.json
expt/assets/trial_sequences/rotated_sequences/expt_info_297_rotated.json
expt/assets/trial_sequences/rotated_sequences/expt_info_413_rotated.json
expt/assets/trial_sequences/rotated_sequences/expt_info_106_rotated.json
expt/assets/trial_sequences/rotated_sequences/expt_info_65_rotated.json
expt/assets/trial_sequences/rotated_sequences/expt_info_385_rotated.json
expt/assets/trial_sequences/rotated_sequences/expt_info_405_rotated.json
expt/assets/trial_sequences/rotated_sequences/expt_info_73_rotated.json
expt/assets/trial_sequences/rotated_sequences/expt_info_110_rotated.json
expt/assets/trial_sequences/rotated_sequences/expt_info_393_rotated.json
expt/assets/trial_sequences/rotated_sequences/expt_inf

In [ ]:
## quick t-test - i.e. for each context, 1-sample ttest on the proportion of trials for which the better axis is horizontal
from scipy.stats import ttest_1samp
for ci, context in enumerate(['row','column']):
    t_stat, p_val = ttest_1samp(ax_balance[:,ci], 0.5)
    print(context,'t-stat:', t_stat, 'p-val:', p_val)
    print('mean proportion of participants  for which the better axis is horizontal:', ax_balance[:,ci].mean())

NameError: name 'ax_balance' is not defined

## Practice Trials

### 1 - practice 2 trials

In [ ]:
## init (DON'T CHANGE THESE)
N = 9
metric = 'cityblock'
known_costs = False
beta_params = {
    'alpha_row': 0.25,
    'beta_row': 0.25,
    'alpha_col': 0.25,
    'beta_col': 0.25
    }
n_cities = 1
n_days = 1
n_trials = 2
n_afc=2
expt = 'AFC'
expt_info = {
    'type': expt,
    'n_afc': n_afc,
    'context': 'column',
    'objective': 'rewards'
}
p=1
save_path = 'expt/assets/trial_sequences/expt_3/practice'
# generate_ppt_sequence(p, n_cities, n_days, n_trials, expt_info.copy(), beta_params, metric, n_afc, N, save_path)

### 2 - illustrate selection

In [ ]:
## init (DON'T CHANGE THESE)
n_cities = 1
n_days = 1
n_trials = 4
p=2
generate_ppt_sequence(p, n_cities, n_days, n_trials, expt_info.copy(), beta_params, metric, n_afc, N, save_path)

,participant,city,context,objective,grid,trial,start_A,start_B,goal_A,goal_B,...,path_B_future_row_overlap,path_A_future_col_overlap,path_B_future_col_overlap,path_A_future_row_and_col_overlap,path_B_future_row_and_col_overlap,path_A_future_rel_overlap,path_B_future_rel_overlap,path_A_future_irrel_overlap,path_B_future_irrel_overlap,better_path
0,2,1,column,rewards,1,1,"(4, 0)","(8, 4)","(0, 4)","(4, 8)",...,13.0,13.0,30.0,0.0,0.0,13.0,30.0,31.0,13.0,a
1,2,1,column,rewards,1,2,"(1, 7)","(3, 8)","(7, 5)","(4, 1)",...,5.0,18.0,29.0,0.0,0.0,18.0,29.0,26.0,5.0,a
2,2,1,column,rewards,1,3,"(1, 1)","(0, 5)","(2, 8)","(7, 6)",...,17.0,17.0,9.0,0.0,0.0,17.0,9.0,9.0,17.0,a
3,2,1,column,rewards,1,4,"(3, 2)","(0, 7)","(1, 8)","(6, 5)",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,b


### 3-4 - col vs row tips

In [ ]:
## init (DON'T CHANGE THESE)
n_cities = 3
n_days = 1
n_trials = 1

## loop through contexts, because we are trying to illustrate each
# for ci, context in enumerate([ 'column', 'row']):
#     p = ci+4
#     expt_info['context'] = context
#     generate_ppt_sequence(p, n_cities, n_days, n_trials, expt_info.copy(), beta_params, metric, n_afc, N, save_path)

### generate sequences under each context HINT: NEED TO CHANGE GENERATE_PPT_SEQUENCE SO THAT CONTEXT IS NOT SHUFFLED

## first, generate column grids 
p = 3
expt_info['context'] = 'column'
expt_info['objective'] = 'rewards'
generate_ppt_sequence(p, n_cities, n_days, n_trials, expt_info.copy(), beta_params, metric, n_afc, N, save_path)

## then, generate row grids
p = 4
expt_info['context'] = 'row'
expt_info['objective'] = 'rewards'
generate_ppt_sequence(p, n_cities, n_days, n_trials, expt_info.copy(), beta_params, metric, n_afc, N, save_path)


,participant,city,context,objective,grid,trial,start_A,start_B,goal_A,goal_B,...,path_B_future_row_overlap,path_A_future_col_overlap,path_B_future_col_overlap,path_A_future_row_and_col_overlap,path_B_future_row_and_col_overlap,path_A_future_rel_overlap,path_B_future_rel_overlap,path_A_future_irrel_overlap,path_B_future_irrel_overlap,better_path
0,4,1,row,rewards,1,1,"(0, 4)","(8, 4)","(4, 8)","(4, 0)",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,b
1,4,2,row,rewards,1,1,"(8, 4)","(0, 4)","(4, 8)","(4, 0)",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,a
2,4,3,row,rewards,1,1,"(0, 4)","(8, 4)","(4, 8)","(4, 0)",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,a


### 5 - practice day (tips)

In [ ]:
## init (DON'T CHANGE THESE)
p=5
n_cities = 1
n_days = 1
n_trials = 4
expt_info['context'] = 'column'
expt_info['objective'] = 'rewards'
generate_ppt_sequence(p, n_cities, n_days, n_trials, expt_info.copy(), beta_params, metric, n_afc, N, save_path)

,participant,city,context,objective,grid,trial,start_A,start_B,goal_A,goal_B,...,path_B_future_row_overlap,path_A_future_col_overlap,path_B_future_col_overlap,path_A_future_row_and_col_overlap,path_B_future_row_and_col_overlap,path_A_future_rel_overlap,path_B_future_rel_overlap,path_A_future_irrel_overlap,path_B_future_irrel_overlap,better_path
0,5,1,column,rewards,1,1,"(8, 4)","(0, 4)","(4, 0)","(4, 8)",...,30.0,30.0,13.0,0.0,0.0,30.0,13.0,15.0,30.0,b
1,5,1,column,rewards,1,2,"(7, 3)","(1, 0)","(0, 2)","(4, 5)",...,20.0,10.0,24.0,0.0,0.0,10.0,24.0,30.0,20.0,a
2,5,1,column,rewards,1,3,"(5, 6)","(0, 1)","(2, 1)","(7, 2)",...,17.0,13.0,9.0,0.0,0.0,13.0,9.0,12.0,17.0,a
3,5,1,column,rewards,1,4,"(3, 0)","(7, 1)","(2, 7)","(0, 0)",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,a


### 6-7 - col vs row tips

In [ ]:
## init (DON'T CHANGE THESE)
n_cities = 3
n_days = 1
n_trials = 1

## loop through contexts, because we are trying to illustrate each
# for ci, context in enumerate([ 'column', 'row']):
#     p = ci+4
#     expt_info['context'] = context
#     generate_ppt_sequence(p, n_cities, n_days, n_trials, expt_info.copy(), beta_params, metric, n_afc, N, save_path)

### generate sequences under each context HINT: NEED TO CHANGE GENERATE_PPT_SEQUENCE SO THAT CONTEXT IS NOT SHUFFLED

## first, generate column grids 
p = 6
expt_info['context'] = 'column'
expt_info['objective'] = 'costs'
generate_ppt_sequence(p, n_cities, n_days, n_trials, expt_info.copy(), beta_params, metric, n_afc, N, save_path)

## then, generate row grids
p = 7
expt_info['context'] = 'row'
expt_info['objective'] = 'costs'
generate_ppt_sequence(p, n_cities, n_days, n_trials, expt_info.copy(), beta_params, metric, n_afc, N, save_path)


,participant,city,context,objective,grid,trial,start_A,start_B,goal_A,goal_B,...,path_B_future_row_overlap,path_A_future_col_overlap,path_B_future_col_overlap,path_A_future_row_and_col_overlap,path_B_future_row_and_col_overlap,path_A_future_rel_overlap,path_B_future_rel_overlap,path_A_future_irrel_overlap,path_B_future_irrel_overlap,better_path
0,7,1,row,costs,1,1,"(8, 4)","(4, 0)","(4, 8)","(0, 4)",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,a
1,7,2,row,costs,1,1,"(4, 8)","(4, 0)","(0, 4)","(8, 4)",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,a
2,7,3,row,costs,1,1,"(4, 0)","(4, 8)","(8, 4)","(0, 4)",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,b


### 8 - practice day (tolls)

In [ ]:
## init (DON'T CHANGE THESE)
p=8
n_cities = 1
n_days = 1
n_trials = 4
expt_info['context'] = 'row'
expt_info['objective'] = 'costs'
generate_ppt_sequence(p, n_cities, n_days, n_trials, expt_info.copy(), beta_params, metric, n_afc, N, save_path)

,participant,city,context,objective,grid,trial,start_A,start_B,goal_A,goal_B,...,path_B_future_row_overlap,path_A_future_col_overlap,path_B_future_col_overlap,path_A_future_row_and_col_overlap,path_B_future_row_and_col_overlap,path_A_future_rel_overlap,path_B_future_rel_overlap,path_A_future_irrel_overlap,path_B_future_irrel_overlap,better_path
0,8,1,row,costs,1,1,"(0, 4)","(8, 4)","(4, 0)","(4, 8)",...,30.0,30.0,12.0,0.0,0.0,13.0,30.0,30.0,12.0,b
1,8,1,row,costs,1,2,"(7, 3)","(6, 7)","(1, 1)","(5, 0)",...,11.0,19.0,30.0,0.0,0.0,27.0,11.0,19.0,30.0,b
2,8,1,row,costs,1,3,"(6, 0)","(8, 2)","(7, 7)","(1, 3)",...,17.0,17.0,9.0,0.0,0.0,8.0,17.0,17.0,9.0,a
3,8,1,row,costs,1,4,"(8, 1)","(7, 0)","(1, 2)","(4, 5)",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,a


## Crop cities

In [ ]:
from PIL import Image
import os

# Define folder path
folder = "expt/assets/cities"

# Loop through images 1.png to 8.png
for i in range(1, 9):
    filename = f"{i}.png"
    filepath = os.path.join(folder, filename)

    # Open the image
    with Image.open(filepath) as img:
        width, height = img.size

        # Crop to top-left 50%
        # scale = 3
        # cropped = img.crop((0, 0, width // scale, height // scale))

        ## crop to bottom left
        scale = 2
        cropped = img.crop((0, height - height // scale, width // scale, height))

        # Save (overwrite or choose a new path)
        # newfilename = f"cropped_{filename}"
        # newfilepath = os.path.join(folder, newfilename)
        # cropped.save(newfilepath)  # Overwrites the original, but 

        # Or, to save to a new folder:
        cropped.save(os.path.join("expt/assets/cities/cropped2", filename))

# do the same with practice1.png and practice2.png
# for i in range(1, 4):
#     filename = f"practice{i}.png"
#     filepath = os.path.join(folder, filename)

#     # Open the image
#     with Image.open(filepath) as img:
#         width, height = img.size

#         # Crop to top-left
#         # scale = 1.5
#         # cropped = img.crop((0, 0, width // scale, height // scale))

#         # or, crop to top right
#         scale = 2
#         cropped = img.crop((width - width // scale, 0, width, height // scale))


#         # Save (overwrite or choose a new path)
#         # newfilename = f"cropped_{filename}"
#         # newfilepath = os.path.join(folder, newfilename)
#         # cropped.save(newfilepath)  # Overwrites the original, but

#         cropped.save(os.path.join("expt/assets/cities/cropped", filename))

